In [11]:
# Mounting Google Drive for later local access.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:

# Project Setup
import os

PROJECT_NAME = "0_potato_project_v1"

PROJECT_DIR = f"/content/drive/MyDrive/{PROJECT_NAME}"
ZIP_PATH    = f"{PROJECT_DIR}/data/potato_raw.zip"

WORK_DIR  = f"/content/{PROJECT_NAME}"
DATA_DIR  = f"{WORK_DIR}/data/raw"
CLONE_DIR = "/content/PlantVillage-Dataset"

# %%bash cells run in a separate process and can't see Python variables.
for n in ["PROJECT_NAME","PROJECT_DIR","ZIP_PATH","WORK_DIR","DATA_DIR","CLONE_DIR"]:
    os.environ[n] = globals()[n]

print("Drive (permanent):", PROJECT_DIR)
print("Local (fast):     ", DATA_DIR)

Drive (permanent): /content/drive/MyDrive/0_potato_project_v1
Local (fast):      /content/0_potato_project_v1/data/raw


In [13]:
%%bash
# Project Directory
mkdir -p "$PROJECT_DIR"/{data,results,notebooks,scripts}
ls "$PROJECT_DIR"

data
notebooks
results
scripts


In [ ]:
%%bash
## Accessing the Colored Potato Dataset of the Plant Village Original
rm -rf "$CLONE_DIR"
cd /content

# Result: all 54,000 images visible without having downloaded any.
git clone --filter=blob:none --no-checkout --depth 1 \
  https://github.com/spMohanty/PlantVillage-Dataset.git

cd "$CLONE_DIR"
git sparse-checkout init --cone
git sparse-checkout set \
  raw/color/Potato___Early_blight \
  raw/color/Potato___Late_blight \
  raw/color/Potato___healthy

git checkout

Your branch is up to date with 'origin/master'.


Cloning into 'PlantVillage-Dataset'...


In [ ]:
## Dumping the Cloned Dataset
%%bash
mkdir -p "$DATA_DIR"

cd "$CLONE_DIR"
git rev-parse HEAD > "$PROJECT_DIR/data/SOURCE_COMMIT.txt"
echo "spMohanty/PlantVillage-Dataset, raw/color, 3 potato classes" >> "$PROJECT_DIR/data/SOURCE_COMMIT.txt"
cat "$PROJECT_DIR/data/SOURCE_COMMIT.txt"

cd "$CLONE_DIR/raw/color"
cp -r Potato___Early_blight "$DATA_DIR/Early_blight"
cp -r Potato___Late_blight  "$DATA_DIR/Late_blight"
cp -r Potato___healthy      "$DATA_DIR/Healthy"

rm -rf "$CLONE_DIR"

spMohanty/PlantVillage-Dataset, raw/color, 3 potato classes


bash: line 3: cd: /content/PlantVillage-Dataset: No such file or directory
fatal: not a git repository (or any of the parent directories): .git
bash: line 8: cd: /content/PlantVillage-Dataset/raw/color: No such file or directory
cp: cannot stat 'Potato___Early_blight': No such file or directory
cp: cannot stat 'Potato___Late_blight': No such file or directory
cp: cannot stat 'Potato___healthy': No such file or directory


In [ ]:
# verifying to ensure we have : 2152 total, split 1000 / 152 / 1000.
total = 0
for cls in sorted(os.listdir(DATA_DIR)):
    n = len(os.listdir(f"{DATA_DIR}/{cls}"))
    total += n
    print(f"{cls:15s} {n}")
print(f"{'TOTAL':15s} {total}")

Early_blight    1000
Healthy         152
Late_blight     1000
TOTAL           2152


In [ ]:
# Archiving to Drive
%%bash
cd "$WORK_DIR/data"
zip -r -q "$ZIP_PATH" raw     # 2,152 files into one archive; -q = don't list them all
ls -lh "$ZIP_PATH"            # should be roughly 40-60MB

-rw------- 1 root root 38M Aug 21 10:29 /content/drive/MyDrive/0_potato_project_v1/data/potato_raw.zip
